# Benchmark: SA vs QQA vs CRA-PI-GNN vs CPRA

Head-to-head comparison of all four solver backends shipped with QQA4CO
on the **same** Maximum Independent Set instance, controlling compute
budget where possible.

**TL;DR (typical numbers on a 200-node 3-regular MIS, CPU):**

| backend | best $|S|$ | wall-clock | strength |
|---------|-----------|------------|----------|
| `qqa.simulated_annealing` | medium | shortest | baseline; no learning |
| `qqa.anneal` (QQA) | high | medium | parallel-replica + diversity term |
| `qqa.pignn.train_cra_pi_gnn` | high | longest | CRA-style annealing on a GCN |
| `qqa.pignn.train_cpra_pi_gnn` | highest | medium-long | R replicas in one forward |

Run on **GPU** for the most informative comparison. CPU also works but the
PyG paths are slow — the 'CPRA wins' regime appears at scale.

In [ ]:
import time

import matplotlib.pyplot as plt
import networkx as nx
import torch

import qqa

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device = {DEVICE}")
if DEVICE == "cuda":
    qqa.enable_tf32()
qqa.fix_seed(0)

## 1. Build a single MIS instance

We use a 200-node random 3-regular graph. Same `seed=0`, same `penalty=2`
for every solver — so any difference in objective is purely algorithmic.

In [ ]:
N = 200
g = nx.random_regular_graph(d=3, n=N, seed=0)
problem = qqa.MaximumIndependentSet(g, penalty=2.0, device=DEVICE)
print(f"|V|={g.number_of_nodes()}  |E|={g.number_of_edges()}")

## 2. Compute budget

We give every solver the same `num_epochs` so wall-clock differences
reflect per-epoch cost, *and* report final objective so you can compare
quality at fixed budget.

In [ ]:
EPOCHS = 2_000
SOL_SIZE = 128

## 3. Simulated Annealing

GPU-parallel single-spin Glauber-like Metropolis (QUBO fast path).

In [ ]:
qqa.fix_seed(0)
t0 = time.time()
sa_result = qqa.simulated_annealing(
    problem,
    sol_size=SOL_SIZE,
    num_sweeps=EPOCHS,
    beta_start=0.1,
    beta_end=20.0,
    beta_schedule="geometric",
    seed=0,
    device=DEVICE,
    verbose=False,
)
sa_time = time.time() - t0
print(f"SA   : best_obj={sa_result.best_obj:.2f}  |S|={int(sa_result.best_sol.sum().item())}  time={sa_time:.2f}s")

## 4. QQA (parallel-replica annealing)

Continuous relaxation + AdamW + bias-gain schedule + diversity term.

In [ ]:
qqa.fix_seed(0)
t0 = time.time()
qqa_result = qqa.anneal(
    problem,
    sol_size=SOL_SIZE,
    num_epochs=EPOCHS,
    learning_rate=1.0,
    div_param=0.0,
    device=DEVICE,
    verbose=False,
)
qqa_time = time.time() - t0
print(f"QQA  : best_obj={qqa_result.best_obj:.2f}  |S|={int(qqa_result.best_sol.sum().item())}  time={qqa_time:.2f}s")

## 5. CRA-PI-GNN (single replica)

GCN backbone + CRA-style penalty annealing. Skip cell if PyG is not
installed (run `pip install qqa[pignn]` to enable).

In [ ]:
try:
    from qqa.pignn import train_cra_pi_gnn
    qqa.fix_seed(0)
    t0 = time.time()
    cra_result = train_cra_pi_gnn(
        problem,
        num_epochs=EPOCHS,
        learning_rate=1e-4,
        device=DEVICE,
        verbose=False,
        seed=0,
    )
    cra_time = time.time() - t0
    print(f"CRA  : best_obj={cra_result.best_obj:.2f}  |S|={int(cra_result.best_sol.sum().item())}  time={cra_time:.2f}s")
except (ImportError, ModuleNotFoundError) as e:
    print(f"CRA-PI-GNN skipped: {e}")
    cra_result = None
    cra_time = None

## 6. CPRA (R parallel replicas, one forward)

Same compute budget, but R=4 replicas trained jointly with a small
variation-diversification penalty.

In [ ]:
try:
    from qqa.pignn import train_cpra_pi_gnn
    qqa.fix_seed(0)
    t0 = time.time()
    cpra_result = train_cpra_pi_gnn(
        problem,
        num_replicas=4,
        vari_param=0.1,
        num_epochs=EPOCHS,
        learning_rate=1e-4,
        device=DEVICE,
        verbose=False,
        seed=0,
    )
    cpra_time = time.time() - t0
    print(f"CPRA : best_obj={cpra_result.best_obj:.2f}  |S|={int(cpra_result.best_sol.sum().item())}  time={cpra_time:.2f}s")
except (ImportError, ModuleNotFoundError) as e:
    print(f"CPRA skipped: {e}")
    cpra_result = None
    cpra_time = None

## 7. Summary table

In [ ]:
rows = [
    ("SA", sa_result.best_obj, int(sa_result.best_sol.sum().item()), sa_time),
    ("QQA", qqa_result.best_obj, int(qqa_result.best_sol.sum().item()), qqa_time),
]
if cra_result is not None:
    rows.append(("CRA-PI-GNN", cra_result.best_obj, int(cra_result.best_sol.sum().item()), cra_time))
if cpra_result is not None:
    rows.append(("CPRA",       cpra_result.best_obj, int(cpra_result.best_sol.sum().item()), cpra_time))

print(f"{'backend':<12} {'best_obj':>10} {'|S|':>6} {'time(s)':>10}")
print("-" * 42)
for name, bo, sz, t in rows:
    print(f"{name:<12} {bo:>10.2f} {sz:>6d} {t:>10.2f}")

## 8. Convergence curves

How quickly does each method drive `best_obj` down?

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(sa_result.history["best_obj"], label="SA", linewidth=2, alpha=0.85)
ax.plot(qqa_result.history["best_obj"], label="QQA", linewidth=2, alpha=0.85)
if cra_result is not None and cra_result.history.get("loss"):
    ax.plot(cra_result.history["loss"], label="CRA-PI-GNN (continuous loss)", linewidth=1, alpha=0.7)
if cpra_result is not None and cpra_result.history.get("per_replica_obj"):
    cpra_best = [min(row) for row in cpra_result.history["per_replica_obj"]]
    ax.plot(cpra_best, label="CPRA (best replica)", linewidth=2, alpha=0.85)

ax.set_xlabel("epoch / sweep")
ax.set_ylabel("objective (lower is better)")
ax.set_title(f"MIS on 3-regular N={N}: convergence comparison")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 9. When to use which?

* **SA** — when you need a *baseline* ("is the fancy method actually
  helping?") or when the problem is tiny and SA's lack of learning
  overhead wins.
* **QQA** — sweet spot for medium-sized QUBO/spin problems
  ($N \lesssim 10^4$). Parallel replicas + diversity term + no GNN
  training overhead.
* **CRA-PI-GNN** — when the problem has obvious *graph structure* (MIS,
  MaxCut, MaxClique) and you can afford the GCN training cost. The GNN
  inductive bias often beats QQA on hard sparse instances.
* **CPRA** — same regime as CRA-PI-GNN but you also need a *diverse pool*
  of solutions (not just the best one) — e.g. for downstream re-ranking,
  ensembling, or ML-coupled pipelines.